# Email Triage RL — GRPO Training (Colab)

**OpenEnv Hackathon 2026 — Team Ctrl-Alt-Defeat**  
Haraprasad Hota · Subhendu Samal · Ashutosh Panigrahi

This notebook trains `Qwen/Qwen2.5-3B-Instruct` to triage emails using **GRPO** (Group Relative Policy Optimisation) via TRL + Unsloth. It connects to the Email Triage RL environment, scores rollouts with **6 independent verifier functions**, and updates the LoRA adapter towards higher-reward completions.

**Reward functions** (imported directly from `train.py` — single source of truth, identical to the production server):
1. `reward_format` — valid JSON + correct email_id (gate signal)
2. `reward_classification` — semantic distance scoring
3. `reward_priority` — graduated 0/±0.4/±1.0
4. `reward_routing` — semantic department distance
5. `reward_escalation` — F1-style TP/TN/FP/FN
6. `reward_response` — keyword coverage with length penalty

**What this notebook produces**
- Baseline-vs-trained scores on all 3 tasks (reproducible, seed=99)
- W&B run with reward curves
- LoRA adapter pushed to `Hk4crprasad/email-triage-grpo`
- Comparison plots saved to `/content/plots/`

**Runtime**: T4 ~45 min · A100 ~12 min

## 1. Install dependencies

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth
!pip install "trl>=0.12.0" datasets accelerate "peft>=0.12.0" bitsandbytes wandb
!pip install "openenv-core>=0.2.0"

In [ ]:
# Pull the environment repo (contains server/, models.py, train.py, etc.)
import os, sys
REPO = '/content/email-triage-env'
if not os.path.isdir(REPO):
    !git clone https://huggingface.co/spaces/Hk4crprasad/email-triage-env $REPO
sys.path.insert(0, REPO)
os.chdir(REPO)
print(f'Working dir: {os.getcwd()}')

In [ ]:
import torch
assert torch.cuda.is_available(), 'GRPO training needs a GPU. Switch runtime → Hardware → T4/A100.'
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} · {props.total_memory / 1e9:.1f} GB · CUDA {torch.version.cuda}')

## 2. Load Qwen2.5-3B-Instruct with Unsloth (4-bit + LoRA r=32)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
LORA_RANK  = 32
MAX_SEQ    = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ,
    load_in_4bit    = True,
    fast_inference  = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r                = LORA_RANK,
    target_modules   = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha       = LORA_RANK * 2,
    lora_dropout     = 0.05,
    bias             = 'none',
    use_gradient_checkpointing = 'unsloth',
)
print(f'✅ {MODEL_NAME} loaded · LoRA r={LORA_RANK} · 4-bit')

## 3. Build the curriculum dataset from the environment

We import `build_dataset` and `format_email_prompt` directly from `train.py` so the prompts the trainer sees are byte-identical to what the production server generates.

In [ ]:
from datasets import Dataset
from train import (
    SYSTEM_PROMPT,
    build_dataset,
    format_email_prompt,
    REWARD_FUNCTIONS,            # all 6 verifiers — same as production
    _parse_action,
)

raw   = build_dataset(['easy', 'medium', 'hard'], seed=42)
ds    = Dataset.from_list(raw)
by_t  = {t: ds.filter(lambda x, t=t: x['task_id'] == t) for t in ['easy','medium','hard']}

print(f'Total: {len(ds)} prompts · easy={len(by_t["easy"])} · medium={len(by_t["medium"])} · hard={len(by_t["hard"])}')
print('\n--- sample prompt (first 500 chars) ---')
print(ds[0]['prompt'][1]['content'][:500])

In [ ]:
# Sanity-check the 6 reward functions against a perfect completion.
perfect = '{"email_id":"%s","category":"%s","priority":%d,"department":"%s","escalate":false}' % (
    raw[0]['email_id'], raw[0]['gt_category'], raw[0]['gt_priority'], raw[0]['gt_department']
)
kw = {k: [raw[0][k]] for k in raw[0]}
for fn in REWARD_FUNCTIONS:
    print(f'{fn.__name__:<22} → {fn([perfect], **kw)[0]:+.2f}')

## 4. Baseline evaluation (BEFORE training)

Run the untrained Qwen2.5-3B against all 3 tasks on a held-out seed. These numbers are our 0-shot baseline.

In [ ]:
from server.environment import EmailTriageEnvironment
import json, time

EVAL_SEED = 99   # held-out — never seen during training

def run_episode(model, tokenizer, task_id, seed=EVAL_SEED):
    """Run a full episode and return (final_score, dim_scores, step_rewards)."""
    env = EmailTriageEnvironment()
    obs = env.reset(seed=seed, task_id=task_id)
    od  = obs.model_dump()
    step_rewards = []
    while not od.get('done', False) and od.get('emails'):
        email = od['emails'][0]
        prompt = format_email_prompt(email, od.get('task_description', ''))
        msgs = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        inputs = tokenizer.apply_chat_template(
            msgs, tokenize=True, add_generation_prompt=True, return_tensors='pt',
        ).to(model.device)
        with torch.no_grad():
            out = model.generate(
                inputs, max_new_tokens=256,
                do_sample=False, pad_token_id=tokenizer.eos_token_id,
            )
        text = tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True)
        action = _parse_action(text) or {
            'email_id': email['email_id'], 'category': 'general',
            'priority': 3, 'department': 'support',
        }
        obs = env.step(action)
        od  = obs.model_dump()
        step_rewards.append(od.get('step_reward', 0.0))
    g = od.get('metadata', {}).get('grading', {})
    return g.get('final_score', 0.0), g.get('dimension_scores', {}), step_rewards

# — Disable LoRA so we evaluate the BASE model (not the freshly-init adapter noise) —
model.disable_adapter_layers()
FastLanguageModel.for_inference(model)

baseline = {}
t0 = time.time()
for tid in ['easy', 'medium', 'hard']:
    score, dims, _ = run_episode(model, tokenizer, tid)
    baseline[tid] = {'score': score, 'dims': dims}
    print(f'baseline {tid:<6} → {score:.4f}   {dims}')
print(f'\nBaseline eval finished in {time.time()-t0:.1f}s')

# Re-enable LoRA layers + put model back into training mode
model.enable_adapter_layers()
FastLanguageModel.for_training(model)

## 5. GRPO training (curriculum: easy → medium → hard)

We chain three training stages on the same adapter. Each stage continues from the previous stage's weights (Unsloth keeps adapters in-place). The advancement thresholds match `/curriculum`:

| stage | task   | steps | threshold |
|-------|--------|------:|-----------|
| 1     | easy   | 200   | mean reward > 0.60 |
| 2     | medium | 100   | mean reward > 0.50 |
| 3     | hard   | 100   | (final stage)      |

Set `WANDB_API_KEY` in the cell below to log metrics — leave blank to skip W&B.

In [ ]:
# Optional: W&B logging — paste your key or leave empty
import os
os.environ.setdefault('WANDB_PROJECT', 'email-triage-rl')
WANDB_KEY = ''   # ← paste your wandb API key for full reward-curve logging
if WANDB_KEY:
    os.environ['WANDB_API_KEY'] = WANDB_KEY
    REPORT = 'wandb'
else:
    REPORT = 'none'
    print('W&B disabled (no key) — training will still produce in-notebook plots')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

STAGES = [
    ('easy',   200),
    ('medium', 100),
    ('hard',   100),
]
NUM_GENERATIONS = 4
BATCH_SIZE      = 1
GRAD_ACCUM      = 4
LR              = 3e-6
MAX_NEW_TOKENS  = 320

trainers = []   # keep references so we can read log_history per stage
for stage_idx, (tid, steps) in enumerate(STAGES):
    cfg = GRPOConfig(
        output_dir                 = f'/content/checkpoints/email-triage-{tid}',
        num_train_epochs           = 1,
        max_steps                  = steps,
        per_device_train_batch_size= BATCH_SIZE,
        gradient_accumulation_steps= GRAD_ACCUM,
        learning_rate              = LR,
        num_generations            = NUM_GENERATIONS,
        max_new_tokens             = MAX_NEW_TOKENS,
        temperature                = 1.0,
        top_p                      = 0.95,
        max_grad_norm              = 0.1,
        warmup_steps               = 20,
        logging_steps              = 5,
        save_steps                 = 100,
        seed                       = 42,
        report_to                  = REPORT,
        run_name                   = f'grpo-{tid}-qwen2.5-3b' if REPORT == 'wandb' else None,
    )
    print(f'\n{"="*60}\n  STAGE {stage_idx+1}/{len(STAGES)} — {tid.upper()} ({steps} steps)\n{"="*60}')
    trainer = GRPOTrainer(
        model           = model,
        reward_funcs    = REWARD_FUNCTIONS,    # all 6 — same as production server
        args            = cfg,
        train_dataset   = by_t[tid],
        processing_class= tokenizer,
    )
    trainer.train()
    trainers.append((tid, trainer))

    # quick mid-curriculum check after each stage
    FastLanguageModel.for_inference(model)
    s, _, _ = run_episode(model, tokenizer, tid)
    FastLanguageModel.for_training(model)
    print(f'  ✓ stage {tid} done · in-stage eval score = {s:.4f}')

## 6. Final evaluation (AFTER training)

In [ ]:
FastLanguageModel.for_inference(model)

trained = {}
for tid in ['easy', 'medium', 'hard']:
    score, dims, _ = run_episode(model, tokenizer, tid)
    trained[tid] = {'score': score, 'dims': dims}
    print(f'trained  {tid:<6} → {score:.4f}   {dims}')

print('\n── Improvement table ──')
print(f"{'task':<8}{'baseline':>10}{'trained':>10}{'Δ':>10}")
for tid in ['easy', 'medium', 'hard']:
    b = baseline[tid]['score']
    t = trained[tid]['score']
    print(f'{tid:<8}{b:>10.4f}{t:>10.4f}{t-b:>+10.4f}')

## 7. Plots — training curve + before/after comparison

In [ ]:
import matplotlib.pyplot as plt, os
import numpy as np
os.makedirs('/content/plots', exist_ok=True)
plt.rcParams['figure.dpi'] = 120

# ── Plot 1: training curve (concatenated across stages) ──
fig, ax = plt.subplots(figsize=(10, 4))
global_step = 0
colors = {'easy':'#4caf50', 'medium':'#ff9800', 'hard':'#f44336'}
for tid, trainer in trainers:
    hist  = trainer.state.log_history
    steps = [h['step'] + global_step for h in hist if 'loss' in h]
    rew   = [h.get('reward', np.nan) for h in hist if 'loss' in h]
    ax.plot(steps, rew, color=colors[tid], linewidth=1.8, label=f'{tid} reward')
    if steps:
        ax.axvspan(steps[0], steps[-1], alpha=0.06, color=colors[tid])
        global_step = steps[-1]
ax.axhline(0.6, ls='--', color='gray', alpha=0.6, label='easy→medium threshold')
ax.axhline(0.5, ls=':',  color='gray', alpha=0.6, label='medium→hard threshold')
ax.set_xlabel('global training step')
ax.set_ylabel('mean reward')
ax.set_title('GRPO training reward — curriculum stages')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('/content/plots/training_curve.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 2: baseline vs trained (final score per task) ──
tasks = ['easy', 'medium', 'hard']
x = np.arange(len(tasks))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
b_scores = [baseline[t]['score'] for t in tasks]
t_scores = [trained[t]['score'] for t in tasks]
bars1 = ax.bar(x - w/2, b_scores, w, label='Baseline (0-shot)', color='#90a4ae')
bars2 = ax.bar(x + w/2, t_scores, w, label='After GRPO',         color='#1e88e5')
for bars in (bars1, bars2):
    for b in bars:
        h = b.get_height()
        ax.text(b.get_x()+b.get_width()/2, h+0.01, f'{h:.2f}', ha='center', fontsize=9)
ax.set_xticks(x); ax.set_xticklabels(tasks)
ax.set_ylabel('Episode final score (0–1)')
ax.set_title('Email Triage — baseline vs GRPO (Qwen2.5-3B-Instruct)')
ax.set_ylim(0, 1.05)
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/plots/score_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── Plot 3: per-dimension gain on medium task ──
med_dims = sorted(set(baseline['medium']['dims']) | set(trained['medium']['dims']))
b_vals = [baseline['medium']['dims'].get(d, 0.0) for d in med_dims]
t_vals = [trained['medium']['dims'].get(d, 0.0) for d in med_dims]
x = np.arange(len(med_dims))
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - 0.2, b_vals, 0.4, label='Baseline', color='#90a4ae')
ax.bar(x + 0.2, t_vals, 0.4, label='Trained',  color='#1e88e5')
ax.set_xticks(x); ax.set_xticklabels(med_dims, rotation=20)
ax.set_ylabel('Accuracy / score')
ax.set_title('Per-dimension breakdown (medium task)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('/content/plots/dimension_breakdown.png', bbox_inches='tight')
plt.show()

## 8. Save adapter + (optional) push to HF Hub

We **never merge** the LoRA into the 4-bit base — that path is known to corrupt weights. We save the adapter only.

In [ ]:
ADAPTER_DIR = '/content/email-triage-grpo-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'✅ Adapter saved → {ADAPTER_DIR}')
!du -sh $ADAPTER_DIR

In [ ]:
# Push to Hub — set your token + target repo, then uncomment the push lines.
HF_TOKEN = ''   # ← paste a write-token from https://huggingface.co/settings/tokens
REPO_ID  = 'Hk4crprasad/email-triage-grpo'

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    model.push_to_hub(REPO_ID, token=HF_TOKEN)
    tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN)
    print(f'✅ Adapter pushed → https://huggingface.co/{REPO_ID}')
else:
    print('⚠ No HF_TOKEN set — skipping push. Adapter remains in /content.')

## 9. Summary

What this notebook just demonstrated end-to-end:

1. **Built the dataset directly from the OpenEnv environment** — no static CSV; prompts come from `email_generator.py`
2. **Trained on a curriculum** — easy → medium → hard, 400 GRPO steps total
3. **Used the same 6 reward functions as the production server** (`from train import REWARD_FUNCTIONS`) — guarantees train/eval consistency
4. **Evaluated on a held-out seed (99)** before *and* after training, on all 3 tasks
5. **Saved comparison plots** to `/content/plots/training_curve.png`, `score_comparison.png`, `dimension_breakdown.png`
6. **Pushed the LoRA adapter** to `Hk4crprasad/email-triage-grpo` (43 MB, mergeable into Qwen2.5-3B-Instruct)

Use the trained adapter:

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-3B-Instruct', device_map='auto')
model = PeftModel.from_pretrained(base, 'Hk4crprasad/email-triage-grpo')
tok   = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-3B-Instruct')
```

Or hit the live env directly:
```bash
curl -X POST https://hk4crprasad-email-triage-env.hf.space/reset \
  -H 'Content-Type: application/json' -d '{"task_id":"easy","seed":42}'
```

**Links** · [HF Space](https://huggingface.co/spaces/Hk4crprasad/email-triage-env) · [Adapter](https://huggingface.co/Hk4crprasad/email-triage-grpo) · [GitHub](https://github.com/hk4crprasad/my-env) · [Blog](https://huggingface.co/blog/Hk4crprasad/email-triage-grpo-blog)